# 2. Research v5 — GPU model comparison

First complete the required index step in notebook 1. Reuse your existing schema-v4 CSV in Drive. No raw-data upload or new CSV preparation is needed.

1. Select **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU), then Save.
2. Run cells in order, approve Drive mounting, and leave `USE_TEXT=False` for the first experiment.
3. Keep the runtime connected. Completed model members are checkpointed to Drive. After a disconnect, choose GPU again and rerun all cells; unchanged completed members resume.

Seven strategies compare multiple balanced samples, recent history, XGBoost, CatBoost, a neural network, separate 10-K/10-Q models, and all controls with equal total class weight. An optional text experiment adds two strategies. **No CPU training fallback.** PyTorch uses Colab's installed CUDA build.

This is an attempt to improve performance, not a promise of 80–85%. The 2024Q4–2025Q3 cohort was already inspected, so these are development comparisons. Thresholds use 2023 calibration labels only; both balanced and full-population results are saved.

In [ ]:
import torch
import subprocess
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then rerun."
print("CUDA:", torch.version.cuda, "GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
%pip -q install "xgboost>=2.1,<4" "catboost>=1.2.7,<2" "scikit-learn>=1.5,<2" "pandas>=2.2,<3" "numpy>=1.26,<3" "pyarrow>=16,<25" "beautifulsoup4>=4.12,<5"

If Colab requests a runtime restart after installation, restart, then run cells again with GPU enabled.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/financial-risk-detector")
DATASET = ROOT/"data/colab_amendment_proxy.csv.gz"
if not DATASET.exists():
    DATASET = ROOT/"colab_amendment_proxy.csv.gz"
assert DATASET.is_file(), f"Prepared schema-v4 CSV missing: {DATASET}"
print("Reusing:", DATASET)

In [ ]:
USE_TEXT = False  # True only after optional evidence collection is complete.
if USE_TEXT:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "sentence-transformers>=3,<6"])

In [ ]:
from pathlib import Path
import sys, importlib
MODULES = Path("/content/research_v5_modules")
MODULES.mkdir(parents=True, exist_ok=True)
SOURCES = {'research_v5_core.py': '"""Point-in-time features and fixed study design; no model fitting or networking."""\nimport hashlib\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nTARGET = \'y_amendment_proxy\'\nSEEDS = (42, 143, 244)\nFOLDS = ((2018, 2020), (2020, 2022))\nRATIOS = (\'accruals_to_assets\', \'net_margin\', \'gross_margin\', \'debt_to_assets\',\n          \'cash_to_assets\', \'working_capital_to_assets\', \'receivables_to_revenue\',\n          \'inventory_to_revenue\', \'operating_cashflow_to_assets\', \'revenue_to_assets\',\n          \'liabilities_to_assets\', \'retained_earnings_to_assets\', \'filing_delay_days\')\nEXCLUDED = {\'adsh\', \'cik\', \'period\', \'period_date\', \'filed\', \'filed_date\', \'filed_year\',\n            \'sic\', \'afs\', \'wksi\', \'days_to_amendment\', TARGET, \'feature_schema_version\',\n            \'label_observation_cutoff\'}\n\n\ndef file_hash(path):\n    result = hashlib.sha256()\n    with open(path, \'rb\') as handle:\n        for chunk in iter(lambda: handle.read(2**20), b\'\'):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef load_data(path):\n    df = pd.read_csv(path, dtype={\'adsh\':str, \'cik\':str}, low_memory=False)\n    required = EXCLUDED | {\'form\', \'fp\', \'sic2\', \'prior_amendment_count\'}\n    if required - set(df):\n        raise ValueError(f\'Missing columns: {sorted(required-set(df))}\')\n    if not df.feature_schema_version.eq(4).all():\n        raise ValueError(\'Use the prepared schema-v4 CSV, not an earlier export.\')\n    df[\'filed_date\'] = pd.to_datetime(df.filed_date, errors=\'raise\')\n    df[\'period_date\'] = pd.to_datetime(df.period_date, errors=\'raise\')\n    df[\'cik\'] = df.cik.astype(str).str.lstrip(\'0\').replace(\'\', \'0\')\n    if not df.adsh.is_unique or df[TARGET].isna().any() or not set(df[TARGET]) <= {0,1}:\n        raise ValueError(\'Invalid labels or duplicate accessions.\')\n    cutoff = pd.to_datetime(df.label_observation_cutoff)\n    if cutoff.nunique() != 1 or cutoff.iloc[0] < pd.Timestamp(\'2026-09-30\'):\n        raise ValueError(\'Require label observation through 2026-09-30.\')\n    if not ((cutoff-df.filed_date).dt.days >= 365).all():\n        raise ValueError(\'An outcome has less than 365 days of follow-up.\')\n    if not set(df.form) <= {\'10-K\',\'10-Q\'}:\n        raise ValueError(\'Predictors must come from original 10-K/10-Q filings.\')\n    df[\'sic2\'] = df.sic2.fillna(\'unknown\').astype(str)\n    return df.reset_index(drop=True)\n\n\ndef balanced(part, seed=42):\n    pos, neg = part[part[TARGET].eq(1)], part[part[TARGET].eq(0)]\n    if pos.empty:\n        raise ValueError(\'Cohort has zero positives.\')\n    controls = []\n    for (year, form), group in pos.groupby([\'filed_year\',\'form\']):\n        pool = neg[neg.filed_year.eq(year) & neg.form.eq(form)]\n        if len(pool) < len(group):\n            raise ValueError(f\'Insufficient controls: {year}, {form}\')\n        controls.append(pool.sample(len(group), replace=False, random_state=seed))\n    return pd.concat([pos, *controls]).sample(frac=1, random_state=seed)\n\n\ndef date_part(df, start, end):\n    return df[df.filed_date.between(pd.Timestamp(start),pd.Timestamp(end))]\n\n\ndef training_pool(df, fit_year, window=8):\n    return date_part(df, f\'{max(2009,fit_year-window+1)}-01-01\', f\'{fit_year}-12-31\')\n\n\ndef add_mature_history(df):\n    """A prior label is usable only AFTER its complete 365-day outcome window."""\n    result = pd.DataFrame(index=df.index)\n    for group_cols, prefix in [([\'cik\'],\'issuer\'), ([\'cik\',\'form\'],\'issuer_form\')]:\n        count = np.zeros(len(df)); positive = np.zeros(len(df))\n        for _, group in df.groupby(group_cols, sort=False):\n            history = group.sort_values(\'filed_date\')\n            maturity = (history.filed_date + pd.Timedelta(days=365)).to_numpy(\'datetime64[D]\')\n            at = np.searchsorted(maturity, group.filed_date.to_numpy(\'datetime64[D]\'), side=\'left\')\n            prefix_sum = np.r_[0, history[TARGET].to_numpy().cumsum()]\n            count[group.index] = at\n            positive[group.index] = prefix_sum[at]\n        result[f\'{prefix}_mature_count\'] = count\n        result[f\'{prefix}_mature_positive_count\'] = positive\n        # Fixed smoothing constant; never estimated from later outcome labels.\n        result[f\'{prefix}_mature_rate\'] = (positive + 0.4) / (count + 20)\n    return pd.concat([df, result], axis=1)\n\n\ndef add_financial_history(df):\n    columns = [c for c in RATIOS if c in df]\n    # One aggregate per issuer/form/day prevents same-day information leakage.\n    keys = [\'cik\',\'form\',\'filed_date\']\n    days = df.groupby(keys, as_index=False)[columns].mean().sort_values(keys)\n    groups = days.groupby([\'cik\',\'form\'], sort=False)\n    # Compute all ratios together; per-issuer Python callbacks are very slow\n    # across hundreds of thousands of filings on a standard Colab runtime.\n    past = groups[columns].shift()\n    rolling = past.groupby([days.cik,days.form],sort=False).rolling(3,min_periods=1)\n    means = rolling.mean().reset_index(level=[0,1],drop=True)\n    stds = rolling.std().reset_index(level=[0,1],drop=True)\n    derived = []\n    for col in columns:\n        days[f\'past3_mean_{col}\'] = means[col]\n        days[f\'past3_std_{col}\'] = stds[col]\n        derived += [f\'past3_mean_{col}\', f\'past3_std_{col}\']\n    out = df.merge(days[keys+derived], on=keys, how=\'left\', validate=\'many_to_one\')\n    extra = {}\n    for col in columns:\n        extra[f\'deviation_past3_{col}\'] = out[col]-out[f\'past3_mean_{col}\']\n    out = pd.concat([out, pd.DataFrame(extra,index=out.index)], axis=1)\n    # Calendar-year aggregates come solely from the PREVIOUS filing year.\n    for col in columns:\n        stats = df.groupby([\'filed_year\',\'form\',\'sic2\'])[col].agg([\'median\',\'count\']).reset_index()\n        stats[\'filed_year\'] += 1\n        stats.loc[stats[\'count\'] < 20, \'median\'] = np.nan\n        name = f\'prior_year_peer_median_{col}\'\n        stats = stats.rename(columns={\'median\':name})\n        out = out.merge(stats[[\'filed_year\',\'form\',\'sic2\',name]],\n                        on=[\'filed_year\',\'form\',\'sic2\'], how=\'left\', validate=\'many_to_one\')\n        out[f\'peer_deviation_{col}\'] = out[col]-out[name]\n    out[\'loss_with_positive_cashflow\'] = ((out.net_income_to_assets < 0) &\n                                           (out.operating_cashflow_to_assets > 0)).astype(\'int8\')\n    out[\'profit_with_negative_cashflow\'] = ((out.net_income_to_assets > 0) &\n                                            (out.operating_cashflow_to_assets < 0)).astype(\'int8\')\n    out[\'negative_equity\'] = out.xbrl_StockholdersEquity.lt(0).astype(\'int8\')\n    facts = [c for c in df if c.startswith(\'xbrl_\')]\n    out[\'financial_fact_missing_fraction\'] = out[facts].isna().mean(axis=1)\n    return out\n\n\ndef add_event_history(df, events):\n    """Public EDGAR index events, all strictly before the predictor date."""\n    out = pd.DataFrame(index=df.index)\n    events = events.copy()\n    events[\'filed_date\'] = pd.to_datetime(events.filed_date)\n    events[\'cik\'] = events.cik.astype(str).str.lstrip(\'0\').replace(\'\',\'0\')\n    categories = {\'8k\':[\'8-K\',\'8-K/A\'], \'late\':[\'NT 10-K\',\'NT 10-Q\'],\n                  \'10ka\':[\'10-K/A\'], \'10qa\':[\'10-Q/A\']}\n    for name, forms in categories.items():\n        event_groups = {cik:np.sort(g.filed_date.to_numpy(\'datetime64[D]\'))\n                        for cik,g in events[events.form.isin(forms)].groupby(\'cik\')}\n        counts = {n:np.zeros(len(df), dtype=\'int32\') for n in (90,365,730)}\n        gap = np.full(len(df), np.nan)\n        for cik, rows in df.groupby(\'cik\',sort=False):\n            dates = event_groups.get(cik, np.array([],dtype=\'datetime64[D]\'))\n            query = rows.filed_date.to_numpy(\'datetime64[D]\')\n            end = np.searchsorted(dates,query,side=\'left\')\n            for span in counts:\n                counts[span][rows.index] = end-np.searchsorted(dates,query-np.timedelta64(span,\'D\'),side=\'left\')\n            valid = end > 0\n            gap[rows.index[valid]] = (query[valid]-dates[end[valid]-1]).astype(\'timedelta64[D]\').astype(int)\n        for span,values in counts.items():\n            out[f\'sec_{name}_count_{span}d\'] = values\n        out[f\'sec_{name}_days_since\'] = gap\n    return pd.concat([df,out],axis=1)\n\n\ndef threshold_choice(y, probability):\n    """Maximise the worst of precision/recall/F1/accuracy on CALIBRATION only."""\n    y = np.asarray(y, dtype=int); p = np.asarray(probability, dtype=float)\n    if len(y) == 0 or len(np.unique(y)) < 2 or not np.isfinite(p).all():\n        raise ValueError(\'Threshold selection requires finite scores and both classes.\')\n    order = np.argsort(-p, kind=\'stable\'); ys = y[order]; ps = p[order]\n    ends = np.r_[np.flatnonzero(ps[:-1] != ps[1:]),len(ps)-1]\n    tp = ys.cumsum()[ends]; fp = ends+1-tp\n    recall = tp/y.sum(); precision = tp/(ends+1)\n    f1 = 2*precision*recall/np.maximum(precision+recall,1e-12)\n    accuracy = (tp+(1-y).sum()-fp)/len(y)\n    floor = np.minimum.reduce([precision,recall,f1,accuracy])\n    i = int(np.argmax(floor))\n    return {\'threshold\':float(ps[ends[i]]), \'calibration_min_metric\':float(floor[i]),\n            \'calibration_all_metrics_ge_80\':bool(np.max(floor) >= .80),\n            \'calibration_all_metrics_ge_85\':bool(np.max(floor) >= .85)}\n\n\ndef text_manifest(df):\n    parts = []\n    for fit_year in (2018,2020,2021):\n        pool = training_pool(df,fit_year)\n        parts += [balanced(pool,seed) for seed in SEEDS]\n    for start,end in [(\'2020-01-01\',\'2020-12-31\'),(\'2022-01-01\',\'2022-12-31\'),\n                      (\'2023-01-01\',\'2023-09-30\'),(\'2024-10-01\',\'2025-09-30\')]:\n        parts.append(date_part(df,start,end))\n    return pd.concat(parts).drop_duplicates(\'adsh\')[[\'adsh\',\'cik\',\'form\',\'filed_date\']].sort_values(\'adsh\')\n', 'sec_v5_evidence.py': '"""Public SEC evidence: complete quarterly indexes and optional ORIGINAL text.\n\nOnly used for predictor enrichment. Existing proxy outcome labels are retained.\nNo contact identity is stored. One sequential client, at most two requests/sec.\n"""\nimport gzip\nimport hashlib\nimport io\nimport importlib.metadata\nimport json\nimport re\nimport sqlite3\nimport time\nfrom pathlib import Path\nfrom contextlib import closing\nfrom urllib.error import HTTPError, URLError\nfrom urllib.request import Request, urlopen\nfrom zipfile import ZipFile\nimport numpy as np\nimport pandas as pd\n\nFORMS = {\'10-K\',\'10-Q\',\'10-K/A\',\'10-Q/A\',\'8-K\',\'8-K/A\',\'NT 10-K\',\'NT 10-Q\'}\nRISK_PATTERNS = {\n    \'weakness\':r\'\\bmaterial weaknesses?\\b\',\n    \'ineffective\':r\'\\b(?:not effective|ineffective)\\b\',\n    \'deficiency\':r\'\\bsignificant deficienc(?:y|ies)\\b\',\n    \'going_concern\':r\'\\bgoing concern\\b\',\n    \'restatement\':r\'\\brestat(?:ement|ements|ed)\\b\',\n    \'related_party\':r\'\\brelated.part(?:y|ies)\\b\',\n    \'estimate\':r\'\\b(?:critical accounting|accounting estimates?)\\b\',\n    \'late\':r\'\\b(?:late filing|untimely|unable to timely)\\b\',\n    \'auditor_change\':r\'\\b(?:auditor resignation|dismissed.*?auditor)\\b\',\n    \'remediation\':r\'\\bremediat(?:ion|e|ed)\\b\',\n    \'controls\':r\'\\b(?:internal controls?|disclosure controls?)\\b\',\n}\n\n\nclass SecClient:\n    def __init__(self, user_agent):\n        if not re.search(r\'\\S+@\\S+\\.\\S+\', user_agent) or \' \' not in user_agent.strip():\n            raise ValueError(\'SEC_USER_AGENT must contain a real name/organization and contact email.\')\n        self.user_agent = user_agent.strip()\n        self.previous = 0.0\n\n    def get(self, url, limit=80*1024**2, original_form=None):\n        if not url.startswith(\'https://www.sec.gov/Archives/\'):\n            raise ValueError(\'Only public SEC archive URLs are allowed.\')\n        for attempt in range(4):\n            time.sleep(max(0, .5-(time.monotonic()-self.previous)))\n            self.previous = time.monotonic()\n            try:\n                request = Request(url, headers={\'User-Agent\':self.user_agent,\'Accept-Encoding\':\'identity\'})\n                with urlopen(request, timeout=90) as response:\n                    chunks=[]; size=0\n                    while True:\n                        block=response.read(64*1024)\n                        if not block:\n                            break\n                        chunks.append(block); size+=len(block)\n                        if size > limit:\n                            raise ValueError(\'SEC response exceeded the configured size limit.\')\n                        if original_form and b\'</DOCUMENT>\' in block.upper():\n                            joined=b\'\'.join(chunks)\n                            if original_document(joined,original_form) is not None:\n                                return joined\n                    return b\'\'.join(chunks)\n            except HTTPError as exc:\n                if exc.code == 403:\n                    raise RuntimeError(\'SEC denied access (403). Stop requests; use an allowed connection later. No bypass is attempted.\') from exc\n                if exc.code not in (429,500,502,503,504):\n                    raise\n                if attempt == 3:\n                    raise RuntimeError(\'SEC rate/server error persisted. Cached progress is retained; retry later.\') from exc\n                time.sleep(min(60,10*2**attempt))\n            except (URLError,TimeoutError) as exc:\n                if attempt == 3:\n                    raise RuntimeError(\'Network unavailable; cached progress is retained.\') from exc\n                time.sleep(3*2**attempt)\n\n\ndef parse_master(raw):\n    records=[]\n    for line in raw.decode(\'latin-1\').splitlines():\n        fields=line.split(\'|\')\n        if len(fields) != 5 or not fields[0].isdigit() or fields[2] not in FORMS:\n            continue\n        cik,_,form,date,path=fields\n        match=re.fullmatch(r\'edgar/data/(\\d+)/(\\d{10}-\\d{2}-\\d{6})\\.txt\',path)\n        if not match or int(match[1]) != int(cik):\n            continue\n        records.append((str(int(cik)),form,date,match[2],path))\n    if not records:\n        raise ValueError(\'SEC index did not contain expected filing records.\')\n    frame=pd.DataFrame(records,columns=[\'cik\',\'form\',\'filed_date\',\'adsh\',\'archive_path\'])\n    frame[\'filed_date\']=pd.to_datetime(frame.filed_date,errors=\'raise\')\n    return frame.drop_duplicates([\'cik\',\'adsh\'])\n\n\ndef load_sec_history(cache, client=None):\n    cache=Path(cache); cache.mkdir(parents=True,exist_ok=True)\n    parts=[]\n    # Predictor dates end in 2025Q3. Later index events are unnecessary.\n    for year in range(2009,2026):\n        for quarter in range(1,5):\n            if year==2025 and quarter>3:\n                break\n            path=cache/f\'{year}Q{quarter}.parquet\'\n            if not path.exists():\n                if client is None:\n                    raise FileNotFoundError(f\'Run the evidence notebook first: missing {path.name}\')\n                url=f\'https://www.sec.gov/Archives/edgar/full-index/{year}/QTR{quarter}/master.zip\'\n                blob=client.get(url)\n                with ZipFile(io.BytesIO(blob)) as z:\n                    member=next((n for n in z.namelist() if n.lower()==\'master.idx\'),None)\n                    if member is None:\n                        raise ValueError(\'Unexpected SEC index ZIP layout.\')\n                    part=parse_master(z.read(member))\n                temp=path.with_suffix(\'.part.parquet\')\n                part.to_parquet(temp,index=False); temp.replace(path)\n                print(f\'Cached {year}Q{quarter}: {len(part):,} relevant filings\',flush=True)\n            part=pd.read_parquet(path)\n            parts.append(part)\n    return pd.concat(parts,ignore_index=True).drop_duplicates([\'cik\',\'adsh\'])\n\n\ndef original_document(blob, form):\n    if form not in (\'10-K\',\'10-Q\'):\n        raise ValueError(\'Text predictors must be an original 10-K or 10-Q.\')\n    text=blob.decode(\'utf-8\',errors=\'replace\')\n    for doc in re.findall(r\'<DOCUMENT>(.*?)</DOCUMENT>\',text,flags=re.I|re.S):\n        kind=re.search(r\'<TYPE>\\s*([^\\r\\n<]+)\',doc,flags=re.I)\n        body=re.search(r\'<TEXT>(.*?)</TEXT>\',doc,flags=re.I|re.S)\n        if kind and body and kind[1].strip().upper()==form:\n            return body[1]\n    return None\n\n\ndef compact_text(html):\n    from bs4 import BeautifulSoup\n    soup=BeautifulSoup(html,\'html.parser\')\n    for tag in soup.find_all([\'script\',\'style\',\'ix:header\',\'ix:hidden\']):\n        tag.decompose()\n    plain=re.sub(r\'\\s+\',\' \',soup.get_text(\' \',strip=True))\n    words=len(plain.split())\n    if words<200:\n        return \'\', {\'text_available\':0.0,\'text_word_log\':float(np.log1p(words))}\n    features={\'text_available\':1.0,\'text_word_log\':float(np.log1p(words))}\n    # Compact excerpts are screening windows, not certified SEC item sections.\n    excerpts=[plain[:1000]]\n    for name,pattern in RISK_PATTERNS.items():\n        matches=list(re.finditer(pattern,plain,flags=re.I))\n        features[f\'text_{name}_per_1k\']=len(matches)*1000/max(words,1)\n        for match in matches[:2]:\n            excerpts.append(plain[max(0,match.start()-250):match.end()+500])\n    return \'\\n\'.join(dict.fromkeys(excerpts))[:10000],features\n\n\ndef text_db(path):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    db=sqlite3.connect(path)\n    db.execute(\'CREATE TABLE IF NOT EXISTS texts (adsh TEXT PRIMARY KEY, url TEXT, status TEXT, body BLOB, lexical TEXT, sha256 TEXT)\')\n    db.commit()\n    return db\n\n\ndef collect_text(manifest, db_path, client, max_new=3000):\n    db=text_db(db_path)\n    done={r[0] for r in db.execute(\'SELECT adsh FROM texts\')}\n    pending=manifest[~manifest.adsh.isin(done)]\n    print(f\'Text cache: {len(manifest)-len(pending):,}/{len(manifest):,} attempted; \'\n          f\'{len(pending):,} pending. This session limit: {max_new or "all"}.\',flush=True)\n    try:\n        for i,row in enumerate(pending.itertuples(index=False),1):\n            if max_new and i>max_new:\n                break\n            if not re.fullmatch(r\'\\d{10}-\\d{2}-\\d{6}\',row.adsh):\n                raise ValueError(\'Invalid accession in text manifest.\')\n            url=f\'https://www.sec.gov/Archives/edgar/data/{int(row.cik)}/{row.adsh}.txt\'\n            body=\'\'; features={\'text_available\':0.0}; digest=\'\'\n            try:\n                blob=client.get(url,limit=25*1024**2,original_form=row.form)\n                html=original_document(blob,row.form)\n                if html is None:\n                    status=\'original_document_missing\'\n                else:\n                    body,features=compact_text(html)\n                    digest=hashlib.sha256(html.encode()).hexdigest()\n                    status=\'ok\' if body else \'insufficient_text\'\n            except HTTPError as exc:\n                if exc.code != 404:\n                    raise\n                status=\'http_404\'\n            except ValueError as exc:\n                if \'size limit\' not in str(exc):\n                    raise\n                status=\'size_limit\'\n            db.execute(\'INSERT OR REPLACE INTO texts VALUES (?,?,?,?,?,?)\',\n                       (row.adsh,url,status,gzip.compress(body.encode()),json.dumps(features),digest))\n            if i%25==0:\n                db.commit(); print(f\'Text requests completed this session: {i:,}\',flush=True)\n    finally:\n        db.commit(); db.close()\n    return text_coverage(manifest,db_path)\n\n\ndef text_coverage(manifest,db_path):\n    with closing(text_db(db_path)) as db:\n        states=dict(db.execute(\'SELECT adsh,status FROM texts\'))\n    status=manifest.adsh.map(states)\n    report={\'requested\':len(manifest),\'attempted\':int(status.notna().sum()),\n            \'usable\':int(status.eq(\'ok\').sum()),\'pending\':int(status.isna().sum())}\n    print(\'Text coverage:\',report,flush=True)\n    return report\n\n\ndef gpu_text_features(manifest,db_path,cache):\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA required for text embedding.\')\n    coverage=text_coverage(manifest,db_path)\n    if coverage[\'pending\']:\n        raise RuntimeError(\'Text download is incomplete. Resume the evidence notebook, or explicitly set USE_TEXT=False.\')\n    if coverage[\'usable\']/coverage[\'requested\'] < .90:\n        raise RuntimeError(\'Text coverage below 90%; inspect failures before enabling the text experiment.\')\n    from sentence_transformers import SentenceTransformer\n    from huggingface_hub import model_info\n    cache=Path(cache); cache.mkdir(parents=True,exist_ok=True)\n    meta_path=cache/\'encoder.json\'\n    name=\'sentence-transformers/all-MiniLM-L6-v2\'\n    if meta_path.exists():\n        meta=json.loads(meta_path.read_text())\n    else:\n        meta={\'model\':name,\'revision\':model_info(name).sha,\'license\':\'Apache-2.0\',\n              \'source\':f\'https://huggingface.co/{name}\',\'text_extraction_version\':1}\n        meta_path.write_text(json.dumps(meta,indent=2))\n    encoder=SentenceTransformer(meta[\'model\'],revision=meta[\'revision\'],device=\'cuda\',trust_remote_code=False)\n    frames=[]\n    with closing(text_db(db_path)) as db:\n        for start in range(0,len(manifest),1000):\n            ids=manifest.adsh.iloc[start:start+1000].tolist()\n            raw=[db.execute(\'SELECT body,lexical,sha256 FROM texts WHERE adsh=?\',(a,)).fetchone() for a in ids]\n            packages={n:importlib.metadata.version(n) for n in\n                      (\'torch\',\'sentence-transformers\',\'transformers\')}\n            key=hashlib.sha256(json.dumps([ids,[r[2] for r in raw],meta,packages],sort_keys=True).encode()).hexdigest()[:24]\n            path=cache/f\'{key}.parquet\'\n            if path.exists():\n                frames.append(pd.read_parquet(path)); continue\n            pieces=[]; owners=[]; lex=[]\n            for i,(body,features,_) in enumerate(raw):\n                lex.append(json.loads(features)); words=gzip.decompress(body).decode().split()\n                for offset in range(0,min(len(words),8*140),140):\n                    pieces.append(\' \'.join(words[offset:offset+140])); owners.append(i)\n            vectors=np.zeros((len(ids),384),dtype=np.float32); counts=np.zeros(len(ids))\n            if pieces:\n                encoded=encoder.encode(pieces,batch_size=64,show_progress_bar=True,\n                                       convert_to_numpy=True,normalize_embeddings=True)\n                np.add.at(vectors,np.asarray(owners),encoded)\n                np.add.at(counts,np.asarray(owners),1)\n                vectors/=np.maximum(counts[:,None],1)\n            frame=pd.concat([pd.DataFrame({\'adsh\':ids}),pd.DataFrame(lex),\n                             pd.DataFrame(vectors,columns=[f\'text_emb_{j}\' for j in range(384)])],axis=1)\n            temporary=path.with_suffix(\'.part.parquet\')\n            frame.to_parquet(temporary,index=False); temporary.replace(path)\n            frames.append(frame)\n    del encoder\n    torch.cuda.empty_cache()\n    return pd.concat(frames,ignore_index=True).fillna(0)\n\n\nif __name__==\'__main__\':\n    import argparse\n    import getpass\n    import os\n    import research_v5_core as core\n    parser=argparse.ArgumentParser(description=\'Cache public SEC evidence without fitting any model.\')\n    parser.add_argument(\'--root\',default=\'.\')\n    parser.add_argument(\'--text\',action=\'store_true\',help=\'Also download optional original-filing excerpts.\')\n    parser.add_argument(\'--max-new\',type=int,default=3000,help=\'Text requests this session; 0 means all pending.\')\n    args=parser.parse_args()\n    if args.max_new<0:\n        parser.error(\'--max-new must be nonnegative\')\n    root=Path(args.root)\n    client=SecClient(os.environ.get(\'SEC_USER_AGENT\') or getpass.getpass(\'Real SEC User-Agent (name/organization and email): \'))\n    events=load_sec_history(root/\'evidence_v5/indexes\',client)\n    print(f\'Index cache complete: {len(events):,} relevant events.\')\n    if args.text:\n        path=root/\'data/colab_amendment_proxy.csv.gz\'\n        if not path.exists():\n            path=root/\'colab_amendment_proxy.csv.gz\'\n        manifest=core.text_manifest(core.load_data(path))\n        manifest.to_csv(root/\'evidence_v5/text_manifest.csv.gz\',index=False,compression=\'gzip\')\n        collect_text(manifest,root/\'evidence_v5/original_text.sqlite\',client,args.max_new)\n', '14_train_v5_colab.py': '"""GPU-only research comparison. Previously examined cohorts are development data.\n\nNo execution on import. The generated Colab notebook calls run().\n"""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport json\nimport time\nimport numpy as np\nimport pandas as pd\nimport research_v5_core as core\nimport sec_v5_evidence as evidence\n\n\ndef atomic_json(path, obj):\n    temp=Path(str(path)+\'.part\')\n    temp.write_text(json.dumps(obj,indent=2),encoding=\'utf-8\'); temp.replace(path)\n\n\ndef metric_report(y,p,threshold):\n    from sklearn.metrics import average_precision_score, roc_auc_score, matthews_corrcoef\n    y=np.asarray(y,dtype=int); p=np.asarray(p,dtype=float); pred=p>=threshold\n    tp=int(np.sum(pred & (y==1))); fp=int(np.sum(pred & (y==0)))\n    fn=int(np.sum(~pred & (y==1))); tn=int(np.sum(~pred & (y==0)))\n    precision=tp/max(tp+fp,1); recall=tp/max(tp+fn,1)\n    f1=2*tp/max(2*tp+fp+fn,1); accuracy=(tp+tn)/len(y)\n    return {\'n\':len(y),\'prevalence\':float(y.mean()),\'threshold\':float(threshold),\n            \'accuracy\':accuracy,\'precision\':precision,\'recall\':recall,\'f1\':f1,\n            \'balanced_accuracy\':(recall+tn/max(tn+fp,1))/2,\n            \'pr_auc\':float(average_precision_score(y,p)),\n            \'roc_auc\':float(roc_auc_score(y,p)),\n            \'mcc\':float(matthews_corrcoef(y,pred)), \'tn_fp_fn_tp\':[tn,fp,fn,tp],\n            \'all_four_metrics_ge_80\':bool(min(accuracy,precision,recall,f1)>=.80),\n            \'all_four_metrics_ge_85\':bool(min(accuracy,precision,recall,f1)>=.85),\n            \'always_negative_accuracy\':float(1-y.mean())}\n\n\ndef cluster_intervals(part, p, threshold, draws=400):\n    y=part[core.TARGET].to_numpy(); pred=p>=threshold\n    rows=pd.DataFrame({\'cik\':part.cik.to_numpy(),\'tp\':pred & (y==1),\n                       \'fp\':pred & (y==0),\'tn\':~pred & (y==0),\'fn\':~pred & (y==1)})\n    groups=rows.groupby(\'cik\')[[\'tp\',\'fp\',\'tn\',\'fn\']].sum().to_numpy()\n    rng=np.random.default_rng(2026); values=[]\n    for _ in range(draws):\n        tp,fp,tn,fn=groups[rng.integers(0,len(groups),len(groups))].sum(axis=0)\n        values.append([tp/max(tp+fp,1),tp/max(tp+fn,1),2*tp/max(2*tp+fp+fn,1),\n                       (tp+tn)/max(tp+fp+tn+fn,1)])\n    limits=np.quantile(values,[.025,.975],axis=0)\n    return {name:[float(limits[0,i]),float(limits[1,i])] for i,name in\n            enumerate([\'precision\',\'recall\',\'f1\',\'accuracy\'])}\n\n\ndef candidate_configs(use_text):\n    configs=[\n        dict(name=\'v4_features_reference\',family=\'xgb\',depth=4,bags=1,window=30,base=True,fixed=201),\n        dict(name=\'history_bagged_xgb\',family=\'xgb\',depth=4,bags=3,window=8),\n        dict(name=\'recent_bagged_xgb\',family=\'xgb\',depth=3,bags=3,window=4),\n        dict(name=\'history_bagged_cat\',family=\'cat\',depth=5,bags=3,window=8),\n        dict(name=\'form_specialists\',family=\'xgb\',depth=4,bags=3,window=8,specialists=True),\n        dict(name=\'all_controls_weighted\',family=\'xgb\',depth=4,bags=1,window=8,all_controls=True),\n        dict(name=\'neural_tabular\',family=\'mlp\',depth=0,bags=1,window=8),\n    ]\n    if use_text:\n        configs += [dict(name=\'disclosure_xgb\',family=\'xgb\',depth=4,bags=3,window=8,text=True),\n                    dict(name=\'disclosure_neural\',family=\'mlp\',depth=0,bags=1,window=8,text=True)]\n    return configs\n\n\ndef feature_columns(df, base_columns, config):\n    allowed=base_columns if config.get(\'base\') else list(df.columns)\n    cat=[c for c in (\'form\',\'fp\',\'sic2\') if c in allowed]\n    num=[c for c in allowed if c not in core.EXCLUDED and c not in cat\n         and pd.api.types.is_numeric_dtype(df[c])\n         and (config.get(\'text\',False) or not c.startswith(\'text_\'))]\n    if not num or set(num)&core.EXCLUDED:\n        raise ValueError(\'Invalid model feature list.\')\n    return num,cat\n\n\ndef matrices(train, predict, numeric, categorical):\n    # Category vocabularies and missingness columns are determined by training only.\n    cat=pd.get_dummies(train[categorical].fillna(\'unknown\').astype(str),dtype=\'float32\')\n    columns=cat.columns.tolist()\n    def transform(part):\n        numeric_part=part[numeric].replace([np.inf,-np.inf],np.nan).astype(\'float32\')\n        cats=pd.get_dummies(part[categorical].fillna(\'unknown\').astype(str),dtype=\'float32\').reindex(columns=columns,fill_value=0)\n        return pd.concat([numeric_part,numeric_part.isna().astype(\'float32\').add_prefix(\'missing__\'),cats],axis=1)\n    return transform(train), transform(predict)\n\n\ndef mlp_fit(train_x,y,predict_x,valid_y,folder,seed,iterations=None):\n    import torch\n    from sklearn.metrics import average_precision_score\n    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n    a=train_x.to_numpy(dtype=np.float32); b=predict_x.to_numpy(dtype=np.float32)\n    usable=~np.isnan(a).all(axis=0)\n    median=np.zeros(a.shape[1],dtype=np.float32)\n    median[usable]=np.nanmedian(a[:,usable],axis=0)\n    a=np.where(np.isnan(a),median,a); b=np.where(np.isnan(b),median,b)\n    mean=a.mean(axis=0); std=np.maximum(a.std(axis=0),1e-3)\n    a=np.clip((a-mean)/std,-10,10); b=np.clip((b-mean)/std,-10,10)\n    np.savez_compressed(folder/\'normalization.npz\',median=median,mean=mean,std=std)\n    device=torch.device(\'cuda\')\n    model=torch.nn.Sequential(torch.nn.Linear(a.shape[1],256),torch.nn.LayerNorm(256),\n          torch.nn.GELU(),torch.nn.Dropout(.3),torch.nn.Linear(256,128),\n          torch.nn.GELU(),torch.nn.Dropout(.2),torch.nn.Linear(128,1)).to(device)\n    xa=torch.tensor(a,device=device); ya=torch.tensor(y,dtype=torch.float32,device=device)\n    xb=torch.tensor(b,device=device)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=.01)\n    best=-np.inf; chosen=0; stale=0\n    def probabilities():\n        model.eval()\n        with torch.no_grad():\n            return np.concatenate([torch.sigmoid(model(xb[i:i+2048])).flatten().cpu().numpy()\n                                   for i in range(0,len(xb),2048)])\n    for epoch in range(iterations or 100):\n        model.train(); order=torch.randperm(len(xa),device=device)\n        for start in range(0,len(order),256):\n            indices=order[start:start+256]\n            loss=torch.nn.functional.binary_cross_entropy_with_logits(model(xa[indices]).flatten(),ya[indices])\n            optimizer.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5)\n            optimizer.step()\n        if iterations is None:\n            score=average_precision_score(valid_y,probabilities())\n            if score>best+1e-5:\n                best=score; chosen=epoch+1; stale=0\n                torch.save(model.state_dict(),folder/\'weights.pt\')\n            else:\n                stale+=1\n            if stale>=12:\n                break\n    if iterations is None:\n        model.load_state_dict(torch.load(folder/\'weights.pt\',map_location=device,weights_only=True))\n    else:\n        chosen=iterations; torch.save(model.state_dict(),folder/\'weights.pt\')\n    p=probabilities()\n    del model,xa,ya,xb\n    torch.cuda.empty_cache()\n    return p,chosen,\'cuda\'\n\n\ndef fit_member(config, train, predict, numeric, categorical, folder, seed, fixed=None):\n    """A completed member is resumed only within a source/data fingerprinted run."""\n    folder.mkdir(parents=True,exist_ok=True)\n    signature=hashlib.sha256(json.dumps([config,train.adsh.tolist(),predict.adsh.tolist(),\n                                        numeric,categorical,seed,fixed],sort_keys=True).encode()).hexdigest()\n    meta_path=folder/\'complete.json\'; probability_path=folder/\'probabilities.npy\'\n    if meta_path.exists() and probability_path.exists():\n        meta=json.loads(meta_path.read_text())\n        if meta[\'signature\']==signature:\n            print(f\'Resume completed member: {folder}\',flush=True)\n            return np.load(probability_path),meta[\'iterations\'],meta\n    print(f\'GPU member {folder.name}: {len(train):,} training rows\',flush=True)\n    y=train[core.TARGET].to_numpy()\n    valid_y=predict[core.TARGET].to_numpy() if fixed is None else None\n    if len(np.unique(y))!=2:\n        raise ValueError(\'Both training classes are required.\')\n    started=time.time()\n    if config[\'family\']==\'cat\':\n        from catboost import CatBoostClassifier\n        x=train[numeric+categorical].copy(); xp=predict[numeric+categorical].copy()\n        x[numeric]=x[numeric].replace([np.inf,-np.inf],np.nan)\n        xp[numeric]=xp[numeric].replace([np.inf,-np.inf],np.nan)\n        for c in categorical:\n            x[c]=x[c].fillna(\'unknown\').astype(str); xp[c]=xp[c].fillna(\'unknown\').astype(str)\n        model=CatBoostClassifier(iterations=fixed or 1600,depth=config[\'depth\'],learning_rate=.03,\n               l2_leaf_reg=12,task_type=\'GPU\',devices=\'0\',loss_function=\'Logloss\',\n               random_seed=seed,allow_writing_files=False,verbose=False)\n        opts={}\n        if fixed is None:\n            stop=core.balanced(predict,seed=707)\n            opts={\'eval_set\':(xp.loc[stop.index],stop[core.TARGET]),\'early_stopping_rounds\':120}\n        model.fit(x,y,cat_features=categorical,**opts)\n        count=fixed or model.get_best_iteration()+1\n        p=model.predict_proba(xp)[:,1]\n        gpu=model.get_all_params().get(\'task_type\')\n        if gpu!=\'GPU\':\n            raise RuntimeError(\'CatBoost did not report GPU execution.\')\n        model.save_model(str(folder/\'model.cbm\')); columns=x.columns.tolist()\n    else:\n        x,xp=matrices(train,predict,numeric,categorical); columns=x.columns.tolist()\n        if config[\'family\']==\'xgb\':\n            from xgboost import XGBClassifier\n            positive_weight=float((y==0).sum()/max((y==1).sum(),1)) if config.get(\'all_controls\') else 1.0\n            model=XGBClassifier(n_estimators=fixed or 1600,max_depth=config[\'depth\'],learning_rate=.025,\n                min_child_weight=5,subsample=.85,colsample_bytree=.85,reg_lambda=8,reg_alpha=.2,\n                scale_pos_weight=positive_weight,tree_method=\'hist\',device=\'cuda\',n_jobs=4,\n                objective=\'binary:logistic\',eval_metric=\'aucpr\',\n                early_stopping_rounds=120 if fixed is None else None,random_state=seed)\n            # Balanced validation is used for early stopping; natural scores are reported separately.\n            if fixed is None:\n                stop=core.balanced(predict,seed=707)\n                model.fit(x,y,eval_set=[(xp.loc[stop.index],stop[core.TARGET])],verbose=False)\n                count=int(model.best_iteration)+1\n            else:\n                model.fit(x,y,verbose=False); count=int(fixed)\n            gpu=json.loads(model.get_booster().save_config())[\'learner\'][\'generic_param\'][\'device\']\n            if not gpu.startswith(\'cuda\'):\n                raise RuntimeError(f\'XGBoost fell back to {gpu}; this experiment requires GPU fitting.\')\n            p=model.predict_proba(xp)[:,1]; model.save_model(str(folder/\'model.json\'))\n        else:\n            p,count,gpu=mlp_fit(x,y,xp,valid_y,folder,seed,iterations=fixed)\n    if not np.isfinite(p).all():\n        raise ValueError(\'Model returned nonfinite probabilities.\')\n    meta={\'signature\':signature,\'iterations\':int(count),\'gpu\':gpu,\'seconds\':time.time()-started,\n          \'train_rows\':len(train),\'train_positives\':int(y.sum()),\'columns\':columns,\n          \'numeric\':numeric,\'categorical\':categorical,\'config\':config,\'seed\':seed}\n    temp=folder/\'probabilities.part.npy\'; np.save(temp,p); temp.replace(probability_path)\n    atomic_json(meta_path,meta)\n    return p,int(count),meta\n\n\ndef fit_ensemble(config, df, fit_year, predict, base_columns, folder, fixed=None):\n    pool=core.training_pool(df,fit_year,config[\'window\'])\n    if pool.filed_date.max()+pd.Timedelta(days=365)>=predict.filed_date.min():\n        raise ValueError(\'Training outcome windows overlap the prediction cohort.\')\n    numeric,categorical=feature_columns(df,base_columns,config)\n    predictions=[]; counts=[]; metadata=[]\n    for bag,seed in enumerate(core.SEEDS[:config[\'bags\']]):\n        train=pool if config.get(\'all_controls\') else core.balanced(pool,seed)\n        member=np.full(len(predict),np.nan)\n        groups=[\'10-K\',\'10-Q\'] if config.get(\'specialists\') else [\'both\']\n        for group in groups:\n            tr=train if group==\'both\' else train[train.form.eq(group)]\n            mask=np.ones(len(predict),dtype=bool) if group==\'both\' else predict.form.eq(group).to_numpy()\n            sub=predict.loc[mask]\n            p,count,meta=fit_member(config,tr,sub,numeric,categorical,folder/f\'bag{bag}_{group}\',seed,\n                                     fixed=config.get(\'fixed\',fixed))\n            member[mask]=p; counts.append(count); metadata.append(meta)\n        if not np.isfinite(member).all():\n            raise ValueError(\'A specialist failed to score its cohort.\')\n        predictions.append(member)\n    return np.mean(predictions,axis=0),counts,metadata\n\n\ndef select_ensemble(predictions,fold_parts,population):\n    from sklearn.metrics import average_precision_score\n    def mean_score(names):\n        scores=[]\n        for j,part in enumerate(fold_parts):\n            selected=core.balanced(part,707).index if population==\'balanced\' else part.index\n            mask=part.index.isin(selected)\n            p=np.mean([predictions[name][j] for name in names],axis=0)\n            scores.append(average_precision_score(part.loc[mask,core.TARGET],p[mask]))\n        return float(np.mean(scores))\n    ranked=sorted(predictions,key=lambda name:mean_score([name]),reverse=True)\n    choices=[ranked[:1],ranked[:min(3,len(ranked))]]\n    selected=max(choices,key=mean_score)\n    return {\'members\':selected,\'cv_pr_auc\':mean_score(selected),\n            \'single_model_ranking\':{name:mean_score([name]) for name in ranked}}\n\n\ndef run(root, use_text=False):\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select a Colab GPU runtime. CUDA is required before any model fit.\')\n    root=Path(root); path=root/\'data/colab_amendment_proxy.csv.gz\'\n    if not path.exists():\n        path=root/\'colab_amendment_proxy.csv.gz\'\n    print(\'Confirmed CUDA GPU:\',torch.cuda.get_device_name(0),flush=True)\n    df=core.load_data(path); base_columns=df.columns.tolist()\n    source_hashes=[core.file_hash(__file__),core.file_hash(core.__file__),core.file_hash(evidence.__file__)]\n    index_paths=[root/\'evidence_v5/indexes\'/f\'{year}Q{quarter}.parquet\'\n                 for year in range(2009,2026) for quarter in range(1,5)\n                 if year<2025 or quarter<=3]\n    missing=[p.name for p in index_paths if not p.exists()]\n    if missing:\n        raise FileNotFoundError(f\'Run the evidence notebook first. Missing index quarters: {missing}\')\n    history_hashes=[core.file_hash(p) for p in index_paths]\n    packages=(\'torch\',\'xgboost\',\'catboost\',\'pandas\',\'numpy\',\'scikit-learn\',\'pyarrow\')\n    if use_text:\n        packages+=(\'sentence-transformers\',\'transformers\',\'huggingface-hub\')\n    versions={name:importlib.metadata.version(name) for name in packages}\n    signatures={\'dataset\':core.file_hash(path),\'code\':source_hashes,\'indexes\':history_hashes,\n                \'text\':use_text,\'versions\':versions,\'gpu\':torch.cuda.get_device_name(0)}\n    print(\'Adding strictly historical outcomes, financial trends, peer comparisons and SEC events.\',flush=True)\n    df=core.add_financial_history(df)\n    df=core.add_mature_history(df)\n    history=evidence.load_sec_history(root/\'evidence_v5/indexes\')\n    df=core.add_event_history(df,history)\n    del history\n    text_coverage=None\n    if use_text:\n        manifest=core.text_manifest(df)\n        features=evidence.gpu_text_features(manifest,root/\'evidence_v5/original_text.sqlite\',root/\'evidence_v5/embeddings\')\n        # Missing documents stay in every evaluation cohort with explicit availability.\n        df=df.merge(features,on=\'adsh\',how=\'left\',validate=\'one_to_one\')\n        for c in features:\n            if c!=\'adsh\':\n                df[c]=df[c].fillna(0)\n        signatures[\'encoder\']=core.file_hash(root/\'evidence_v5/embeddings/encoder.json\')\n        signatures[\'text_features\']=hashlib.sha256(pd.util.hash_pandas_object(features,index=False).values.tobytes()).hexdigest()\n        text_coverage=df[df.adsh.isin(manifest.adsh)].groupby([\'filed_year\',core.TARGET]).text_available.agg([\'count\',\'mean\']).reset_index().to_dict(\'records\')\n    digest=hashlib.sha256(json.dumps(signatures,sort_keys=True).encode()).hexdigest()[:16]\n    out=root/\'colab_outputs\'/f\'research_v5_{digest}\'; out.mkdir(parents=True,exist_ok=True)\n    atomic_json(out/\'provenance.json\',{\'signatures\':signatures,\'versions\':versions,\n           \'gpu\':torch.cuda.get_device_name(0),\'text_coverage\':text_coverage,\n           \'evaluation_status\':\'2024Q4-2025Q3 already inspected; development comparison, not an untouched test\',\n           \'labels\':\'Existing FSDS same-period amendment proxy unchanged; master indexes enrich predictors only\',\n           \'history_limit\':\'SEC index snapshots may incorporate later corrections/removals; historical archive starts 2009\',\n           \'sampling\':\'Each bag is 1:1 matched by year/form; all-controls arm balances total class weight\',\n           \'created_utc\':pd.Timestamp.now(tz=\'UTC\').isoformat()})\n    print(\'Resumable run directory:\',out,flush=True)\n    configs=candidate_configs(use_text); cv_predictions={}; leaderboard=[]; counts_by_name={}\n    fold_parts=[core.date_part(df,f\'{year}-01-01\',f\'{year}-12-31\') for _,year in core.FOLDS]\n    from sklearn.metrics import average_precision_score\n    for config in configs:\n        predictions=[]; counts=[]; balanced_scores=[]; natural_scores=[]\n        for (fit_year,valid_year),part in zip(core.FOLDS,fold_parts):\n            print(f\'Fit {config["name"]}: through {fit_year}, validation {valid_year}\',flush=True)\n            p,iterations,_=fit_ensemble(config,df,fit_year,part,base_columns,\n                                        out/\'cv\'/config[\'name\']/str(valid_year))\n            predictions.append(p); counts.extend(iterations)\n            mask=part.index.isin(core.balanced(part,707).index)\n            balanced_scores.append(float(average_precision_score(part.loc[mask,core.TARGET],p[mask])))\n            natural_scores.append(float(average_precision_score(part[core.TARGET],p)))\n        count=int(np.median(counts))\n        eligible=config[\'family\']==\'mlp\' or count>=25\n        row={**config,\'cv_balanced_pr_auc\':float(np.mean(balanced_scores)),\n             \'cv_natural_pr_auc\':float(np.mean(natural_scores)),\'fold_balanced_pr_auc\':balanced_scores,\n             \'fold_natural_pr_auc\':natural_scores,\'iterations\':counts,\'final_iterations\':count,\n             \'eligible\':eligible}\n        leaderboard.append(row); print(row,flush=True)\n        atomic_json(out/\'cv_progress.json\',leaderboard)\n        if eligible:\n            cv_predictions[config[\'name\']]=predictions; counts_by_name[config[\'name\']]=count\n    if not cv_predictions:\n        raise RuntimeError(\'No eligible models survived the CV checks.\')\n    choices={pop:select_ensemble(cv_predictions,fold_parts,pop) for pop in (\'balanced\',\'natural\')}\n    atomic_json(out/\'selected_before_evaluation.json\',choices)\n    cal=core.date_part(df,\'2023-01-01\',\'2023-09-30\')\n    dev=core.date_part(df,\'2024-10-01\',\'2025-09-30\')\n    # All training labels mature before calibration; all calibration labels mature before development.\n    if not (pd.Timestamp(\'2021-12-31\')+pd.Timedelta(days=365)<cal.filed_date.min()\n            and cal.filed_date.max()+pd.Timedelta(days=365)<dev.filed_date.min()):\n        raise ValueError(\'Temporal embargo is insufficient for the 365-day label.\')\n    predict=pd.concat([cal,dev]); scores={}\n    selected_names=set(sum([c[\'members\'] for c in choices.values()],[]))\n    for config in configs:\n        if config[\'name\'] not in selected_names:\n            continue\n        print(\'Fixed-iteration final fit:\',config[\'name\'],flush=True)\n        p,_,_=fit_ensemble(config,df,2021,predict,base_columns,out/\'evaluation_models\'/config[\'name\'],\n                           fixed=counts_by_name[config[\'name\']])\n        scores[config[\'name\']]=p\n    results={\'evaluation_status\':\'Previously inspected development cohort; no pristine final score claimed\',\n             \'target\':\'FSDS same-period 365-day amendment proxy, not a confirmed restatement\',\n             \'selection\':choices,\'cv_leaderboard\':leaderboard,\'text_coverage\':text_coverage}\n    predictions=dev[[\'adsh\',\'cik\',\'form\',\'filed_date\',core.TARGET]].copy()\n    for pop in (\'balanced\',\'natural\'):\n        p=np.mean([scores[name] for name in choices[pop][\'members\']],axis=0)\n        pc,pd_=p[:len(cal)],p[len(cal):]\n        cal_mask=cal.index.isin(core.balanced(cal,43).index) if pop==\'balanced\' else np.ones(len(cal),bool)\n        dev_mask=dev.index.isin(core.balanced(dev,44).index) if pop==\'balanced\' else np.ones(len(dev),bool)\n        selection=core.threshold_choice(cal.loc[cal_mask,core.TARGET],pc[cal_mask])\n        threshold=selection[\'threshold\']\n        results[pop]={\'threshold_selection\':selection,\n                      \'calibration\':metric_report(cal.loc[cal_mask,core.TARGET],pc[cal_mask],threshold),\n                      \'development\':metric_report(dev.loc[dev_mask,core.TARGET],pd_[dev_mask],threshold),\n                      \'issuer_bootstrap_95pct\':cluster_intervals(dev.loc[dev_mask],pd_[dev_mask],threshold)}\n        predictions[f\'{pop}_score\']=pd_; predictions[f\'{pop}_prediction\']=(pd_>=threshold).astype(int)\n        predictions[f\'in_{pop}_evaluation\']=dev_mask\n        # Applying the balanced model to all rows is reported explicitly as well.\n        if pop==\'balanced\':\n            results[\'balanced_model_on_natural_population\']=metric_report(dev[core.TARGET],pd_,threshold)\n    predictions.to_csv(out/\'development_predictions.csv.gz\',index=False,compression=\'gzip\')\n    for year,part in zip([y for _,y in core.FOLDS],fold_parts):\n        j=[y for _,y in core.FOLDS].index(year)\n        exported=part[[\'adsh\',\'cik\',\'form\',\'filed_date\',core.TARGET]].copy()\n        for name in cv_predictions:\n            exported[f\'score_{name}\']=cv_predictions[name][j]\n        exported.to_csv(out/f\'cv_predictions_{year}.csv.gz\',index=False,compression=\'gzip\')\n    pd.DataFrame(leaderboard).to_csv(out/\'cv_leaderboard.csv\',index=False)\n    atomic_json(out/\'metrics.json\',results)\n    print(json.dumps({pop:results[pop] for pop in (\'balanced\',\'natural\')},indent=2))\n    print(\'Finished. Return metrics.json, cv_leaderboard.csv, provenance.json and prediction CSVs from\',out)\n    return out\n\n\nif __name__==\'__main__\':\n    run(\'/content/drive/MyDrive/financial-risk-detector\')\n'}
for name, source in SOURCES.items():
    (MODULES/name).write_text(source, encoding="utf-8")
if str(MODULES) not in sys.path:
    sys.path.insert(0, str(MODULES))
import research_v5_core as core
import sec_v5_evidence as evidence
importlib.reload(core)
importlib.reload(evidence)
print("Embedded source modules ready.")

## Train and evaluate

All model fits explicitly use CUDA. CPU activity during CSV reading, feature preparation, thresholds, and reporting is expected. Evaluation models use iteration counts selected by chronological validation; early stopping never sees the development cohort.

Outputs go to `MyDrive/financial-risk-detector/colab_outputs/research_v5_<fingerprint>/`. Fingerprints include data, code, dependencies, evidence, and GPU type. Changing those starts a new run. A checkpoint resumes complete model members; an interrupted member is retrained. Do not change settings based on development scores and then call that cohort an untouched test.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("training_v5", MODULES/"14_train_v5_colab.py")
training = importlib.util.module_from_spec(spec)
spec.loader.exec_module(training)
OUTPUT = training.run(ROOT, use_text=USE_TEXT)

In [ ]:
import json
import pandas as pd
metrics = json.loads((OUTPUT/"metrics.json").read_text())
summary = pd.DataFrame({name: metrics[name]["development"]
                        for name in ("balanced", "natural")}).T
display(summary[["n", "prevalence", "accuracy", "precision", "recall", "f1",
                 "pr_auc", "roc_auc", "all_four_metrics_ge_80", "all_four_metrics_ge_85"]])
print("Saved to:", OUTPUT)
print("Return metrics.json, cv_leaderboard.csv, provenance.json, and development_predictions.csv.gz.")

## Read the results

`balanced` uses one control per positive. `natural` includes every eligible filing. Read precision, recall, F1, and accuracy together with prevalence and the confusion matrix; high natural accuracy alone is not success. `issuer_bootstrap_95pct` reports uncertainty. No difficult cases are removed to inflate results.

The target remains an FSDS-observed same-period amendment within 365 days. These are academic screening experiments, not confirmed-restatement or fraud determinations. Public SEC access follows the declared-identity and throttling guidance; do not bypass denied access.